# 08. 저장된 예측으로 모델 가치·의사결정 분석

05·07의 **기존 Valid/Test 예측 파일만** 읽습니다. 학습, Feature 재생성, Git 갱신은 하지 않습니다. 기준일 `t`에서 `t+1`~`t+7`의 동일 매장·상품군·목표일을 비교하고, 전주 동일 요일·최근 4주 동일 요일 평균을 기준선으로 둡니다.

Test는 과거 실험에서 이미 확인한 회고적 구간입니다. 여기서 모델 설정을 다시 선택하지 않습니다. 실제 물량 배분·발주 효과는 재고와 공급 자료가 없으므로 측정하지 않습니다.


## Colab 준비

별도의 00 노트북으로 저장소가 My Drive에 준비되어 있어야 합니다. 원본 CSV나 GPU 런타임은 이 분석에 필요하지 않습니다.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
%cd /content/drive/MyDrive/retail-demand-forecasting


### 시각화 패키지

모델을 설치하거나 학습하지 않습니다. 결과 Parquet과 한글 그래프를 읽고 그리는 패키지만 준비합니다.


In [ ]:
%pip install -q seaborn koreanize-matplotlib pyarrow


## Library


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import koreanize_matplotlib

REPO = Path("/content/drive/MyDrive/retail-demand-forecasting")
assert (REPO / "src/retail_forecast/__init__.py").is_file(), REPO
sys.path.insert(0, str(REPO / "src"))

from retail_forecast.comparison import compare_on_common_rows
from retail_forecast.neural_data import KEYS, summarize_neural_predictions

sns.set_theme(style="whitegrid", font="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False


## Utils

공통 행 비교, 기준선 추가, 그룹별 오류 계산에 필요한 짧은 함수만 둡니다.


In [ ]:
def prediction_paths(ml_dir, dl_dir, phase):
    return {
        "LightGBM": ml_dir / f"{phase}_predictions_lightgbm.csv",
        "CatBoost": ml_dir / f"{phase}_predictions_catboost.csv",
        "NHITS": dl_dir / "nhits" / f"{phase}_predictions_nhits.parquet",
        "TCN": dl_dir / "tcn" / f"{phase}_predictions_tcn.parquet",
        "TFT": dl_dir / "tft" / f"{phase}_predictions_tft.parquet",
    }


def compare_with_baselines(paths):
    model_metrics = compare_on_common_rows(paths)
    reference = pd.read_csv(paths["LightGBM"], parse_dates=["date", "target_date"])
    baselines = {
        "전주 동일 요일": "sales_same_weekday_last_week",
        "최근 4주 동일 요일 평균": "sales_same_weekday_4w_mean",
    }
    baseline_metrics = []
    for name, column in baselines.items():
        frame = reference[KEYS + ["target_sales", column]].rename(columns={column: "prediction"})
        metrics, _ = summarize_neural_predictions(frame, name)
        baseline_metrics.append(metrics)
    return pd.concat([model_metrics, *baseline_metrics], ignore_index=True)


def group_scores(frame, groups):
    errors = frame.assign(
        model_absolute=(frame["prediction"] - frame["target_sales"]).abs(),
        baseline_absolute=(frame["sales_same_weekday_4w_mean"] - frame["target_sales"]).abs(),
        model_signed=frame["prediction"] - frame["target_sales"],
    )
    result = errors.groupby(groups, observed=True).agg(
        rows=("target_sales", "size"),
        actual_sales=("target_sales", "sum"),
        model_absolute=("model_absolute", "sum"),
        baseline_absolute=("baseline_absolute", "sum"),
        model_signed=("model_signed", "sum"),
    ).reset_index()
    denominator = result["actual_sales"].replace(0, np.nan)
    result["lightgbm_wape"] = 100 * result["model_absolute"] / denominator
    result["baseline_wape"] = 100 * result["baseline_absolute"] / denominator
    result["gain_pp"] = result["baseline_wape"] - result["lightgbm_wape"]
    result["lightgbm_bias"] = 100 * result["model_signed"] / denominator
    return result.drop(columns=["model_absolute", "baseline_absolute", "model_signed"])


def weekly_predictions(frame):
    keys = ["date", "store_nbr", "family"]
    assert frame.groupby(keys, observed=True)["forecast_horizon"].nunique().eq(7).all()
    return frame.groupby(keys, observed=True)[
        ["target_sales", "prediction", "sales_same_weekday_4w_mean"]
    ].sum(min_count=7).reset_index()


## 결과 경로

05와 07에서 사용한 실행 이름을 그대로 적습니다. 필요한 예측 파일이 없으면 해당 실행의 **결과 파일**을 먼저 준비하세요. 이 노트북은 학습을 대신 실행하지 않습니다.


In [ ]:
ML_RUN_NAME = "full_history_time_split_fixed_i600"
DL_RUN_NAME = "neural_time_split_val_loss"

ml_dir = REPO / "outputs" / ML_RUN_NAME
dl_dir = REPO / "outputs" / DL_RUN_NAME
analysis_dir = REPO / "outputs" / "decision_analysis" / f"{ML_RUN_NAME}__{DL_RUN_NAME}"
analysis_dir.mkdir(parents=True, exist_ok=True)

valid_paths = prediction_paths(ml_dir, dl_dir, "validation")
test_paths = prediction_paths(ml_dir, dl_dir, "test")
missing = [path for path in [*valid_paths.values(), *test_paths.values()] if not path.is_file()]
assert not missing, f"필요한 결과 파일이 없습니다: {missing}"


## 1. 동일한 행에서 모델·기준선 비교

Valid와 Test를 별도로 계산합니다. `compare_on_common_rows`가 다섯 모델의 기준일·목표일·매장·상품군·horizon과 정답이 일치하는지 검사합니다. 양쪽 기준선은 LightGBM 결과 파일에 저장된 같은 행의 과거 판매값을 사용합니다.


In [ ]:
valid_table = compare_with_baselines(valid_paths)
test_table = compare_with_baselines(test_paths)

valid_table.to_csv(analysis_dir / "comparison_with_baselines_valid.csv", index=False)
test_table.to_csv(analysis_dir / "comparison_with_baselines_test.csv", index=False)

overall = pd.concat([
    valid_table.assign(phase="Valid"),
    test_table.assign(phase="Test"),
], ignore_index=True)
overall = overall.loc[
    overall["forecast_horizon"].eq(0),
    ["phase", "model", "scope", "rows", "mae", "rmse", "wape", "bias"],
].sort_values(["phase", "scope", "wape"])
display(overall)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(
    data=overall.loc[overall["phase"].eq("Test")],
    x="wape", y="model", hue="scope", ax=ax,
)
ax.set(xlabel="WAPE (%)", ylabel="", title="동일 Test 행: 모델과 주간 기준선")
fig.tight_layout()
fig.savefig(analysis_dir / "test_wape_comparison.png", dpi=160)
plt.show()


## 2. LightGBM이 기준선보다 나쁜 구간 찾기

`gain_pp = 기준선 WAPE − LightGBM WAPE`입니다. **음수**이면 LightGBM이 최근 4주 평균보다 나쁩니다. `actual_sales`가 작은 집단에서는 WAPE가 불안정할 수 있으므로 행 수와 판매량을 함께 봅니다. 아래 표는 Test를 해석하기 위한 것이며 Test 결과로 파라미터를 조정하지 않습니다.


In [ ]:
test_predictions = pd.read_csv(test_paths["LightGBM"], parse_dates=["date", "target_date"])

horizon_scores = group_scores(test_predictions, ["forecast_horizon"])
family_scores = group_scores(test_predictions, ["family"])
store_scores = group_scores(test_predictions, ["store_nbr"])

for name, table in {
    "horizon": horizon_scores,
    "family": family_scores,
    "store": store_scores,
}.items():
    table.to_csv(analysis_dir / f"test_error_by_{name}.csv", index=False)

display(horizon_scores)
display(family_scores.sort_values("gain_pp").head(12))
display(store_scores.sort_values("gain_pp").head(12))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=horizon_scores, x="forecast_horizon", y="gain_pp", marker="o", ax=axes[0])
sns.lineplot(data=horizon_scores, x="forecast_horizon", y="lightgbm_bias", marker="o", ax=axes[1])
axes[0].axhline(0, color="gray", linewidth=1)
axes[0].set(xlabel="예측 거리 (일)", ylabel="WAPE 개선 (pp)", title="4주 기준선 대비 개선")
axes[1].set(xlabel="예측 거리 (일)", ylabel="Bias (%)", title="LightGBM 과대·과소예측")
fig.tight_layout()
fig.savefig(analysis_dir / "test_horizon_errors.png", dpi=160)
plt.show()


## 3. 수요 규모별 7일 합계 오류

수요 규모는 **기준일에 계산 가능한 최근 4주 동일 요일 예측의 7일 합계**로 나눕니다. Test의 실제 판매량으로 그룹을 만들지 않습니다. 세 구간은 이 실행의 예측 규모 순위에 따른 기술적 구분이며, 완성된 군집 모델은 아닙니다.


In [ ]:
weekly = weekly_predictions(test_predictions)
weekly_family_scores = group_scores(weekly, ["family"])
weekly_store_scores = group_scores(weekly, ["store_nbr"])
weekly_family_scores.to_csv(analysis_dir / "test_7day_error_by_family.csv", index=False)
weekly_store_scores.to_csv(analysis_dir / "test_7day_error_by_store.csv", index=False)

weekly["demand_tier"] = pd.cut(
    weekly["sales_same_weekday_4w_mean"].rank(pct=True, method="average"),
    bins=[0, 1 / 3, 2 / 3, 1],
    labels=["낮음", "중간", "높음"],
    include_lowest=True,
)
tier_scores = group_scores(weekly, ["demand_tier"])
tier_scores.to_csv(analysis_dir / "test_7day_error_by_demand_tier.csv", index=False)
display(tier_scores)
display(weekly_family_scores.sort_values("gain_pp").head(12))
display(weekly_store_scores.sort_values("gain_pp").head(12))


## 4. 주간 검토 후보

LightGBM의 7일 예상 수요와 4주 기준선의 차이로 **검토 후보**를 정렬합니다. 선택 기준에는 Test 정답을 사용하지 않습니다. `target_sales`는 과거 결과를 확인하기 위한 열로만 붙입니다. 재고·공급 가능량이 없으므로 실제 배분량이나 주문량을 제안하지 않습니다.


In [ ]:
weekly["forecast_change"] = weekly["prediction"] - weekly["sales_same_weekday_4w_mean"]
weekly["forecast_change_pct"] = 100 * weekly["forecast_change"] / weekly[
    "sales_same_weekday_4w_mean"
].replace(0, np.nan)

review_columns = [
    "date", "store_nbr", "family", "sales_same_weekday_4w_mean",
    "prediction", "forecast_change", "forecast_change_pct", "target_sales",
]
increasing_demand = weekly.nlargest(20, "forecast_change")[review_columns]
decreasing_demand = weekly.nsmallest(20, "forecast_change")[review_columns]

increasing_demand.to_csv(analysis_dir / "review_increasing_demand.csv", index=False)
decreasing_demand.to_csv(analysis_dir / "review_decreasing_demand.csv", index=False)
display(increasing_demand.head(10))
display(decreasing_demand.head(10))


## 해석 원칙과 다음 검증

- 전체 WAPE 개선과 **매장·상품군별 실패**를 함께 보고합니다. Bias가 양수이면 주간 총수요를 과대예측하는 방향입니다.
- 7일 합계의 오차가 작아도 일별 과대·과소예측이 상쇄될 수 있습니다. 7일 합계 창은 서로 겹치므로 행 수만큼 독립적인 운영 사례를 검증한 것은 아닙니다.
- 모델별 입력 Feature가 동일하지 않아 ML·DL 성능 차이를 구조만의 효과로 해석하지 않습니다.
- 저장된 Test는 이전 실험에서 확인한 **회고적 구간**입니다. 모델 설정을 고정한 뒤 다른 과거 시점의 시간순 평가로 개선이 반복되는지 확인하는 것은 별도 후속 실험입니다.
- 재고·공급·납기와 실제 담당자의 판단 기록이 없으므로 품절 감소, 비용 절감, 최적 발주량은 현재 결과로 검증할 수 없습니다.
